In [ ]:
import requests

# Define a reusable system prompt (set this globally or pass in if dynamic)
task_prompt = (
    "You are an AI assistant verifying whether a location truly has a toll booth. "
    "Given some text scraped from a web page, the expected toll name, and GPS coordinates, "
    "determine if the location is a valid toll plaza. Provide justification."
)

def llm_verify(text, name, coordinates, model="chatgpt"):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": task_prompt},
            {"role": "user", "content": f"""Toll name: {name}
Coordinates: {coordinates}
Web Text:
{text}
Is this a valid toll?"""}
        ],
        "stream": False  # Set True if you want streaming output
    }

    try:
        response = requests.post("http://localhost:11434/api/chat", json=payload)
        response.raise_for_status()
        result = response.json()
        reply = result.get("message", {}).get("content", "")
        print(f"\n🔍 LLM Verified Output for '{name}':\n{reply}\n")
        return reply
    except Exception as e:
        print(f"⚠️ LLM verification failed: {e}")
        return None


In [ ]:
def classify_verification(text, model="llama3"):
    prompt = f"""
Read the following verification explanation and return 1 if it says the toll is invalid or not found.
Return 0 if the toll is real or confirmed.

Text:
{text}

Only respond with 1 or 0.
"""

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False
    }

    try:
        response = requests.post("http://localhost:11434/api/chat", json=payload)
        response.raise_for_status()
        content = response.json().get("message", {}).get("content", "").strip()
        return int(content[0]) if content[0] in "01" else 1  # assume bad = 1
    except Exception as e:
        print(f"⚠️ Classification failed: {e}")
        return 1


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import pandas as pd


In [ ]:

# --- Setup Selenium Headless Chrome ---
def init_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # comment this out to see browser
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=chrome_options)
    return driver



In [ ]:
# --- Google Search and Get Text from Result Page ---
def google_and_scrape(driver, query):
    search_url = f"https://www.google.com/search?q={query.replace(' ', '+')}"
    driver.get(search_url)
    time.sleep(2.5)  # Let page load

    # Grab visible text from search result page
    try:
        text = driver.find_element(By.TAG_NAME, 'body').text
    except Exception as e:
        text = ""
    return text

# --- Main Execution ---
def run_scrape_and_verify(df):
    driver = init_driver()

    for i in range(len(df)):
        name = df['name'].iloc[i]
        lat = df['lat'].iloc[i]
        lon = df['lon'].iloc[i]

        query = f"{name} toll"
        print(f"Searching: {query}")

        text = google_and_scrape(driver, query)

        coordinates = (lat, lon)

        # Send to your LLM verification function (not implemented here)
        llm_verify(text, name, coordinates)  # assumed to exist

    driver.quit()


In [ ]:
def clean_invalid_records(records_df):
    to_drop = []

    for idx, row in records_df.iterrows():
        name = row['toll_name1']
        coords = (row['lat1'], row['lon1'])

        # You can optionally include toll_name2 as well
        verification_text = llm_verify(text="", name=name, coordinates=coords)
        decision = classify_verification(verification_text)

        if decision == 1:
            to_drop.append(idx)

    cleaned_df = records_df.drop(index=to_drop).reset_index(drop=True)
    return cleaned_df
